# 1. LLM

In [4]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI


load_dotenv()

model = ChatOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    model="deepseek-flash",
    temperature=0,
)

# LLM + tools
# model_with_tools = model.bind_tools(tools)

In [ ]:
response = model.invoke(
    [{
        "role": "user",
        "content": "hello, my name is Bob."
    }]
)

print(response.content)

content='Hello Bob, nice to meet you! How can I help you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 37, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 45, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 37}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'a2a765f1-b4f5-42fe-b655-dd856ad39745', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a11513-6817-72f2-bb91-56d817b0404b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 37, 'output_tokens': 61, 'total_tokens': 98, 'input_token_details': {'cache_read': 0}, 'output_tok

# 2. Tools

- file
- shell
- rag
- (memory db)

In [9]:
# file tool
from langchain_community.agent_toolkits import FileManagementToolkit

file_toolkit = FileManagementToolkit(
    root_dir="./workspace",          # 限制工作区
    selected_tools=["read_file", "write_file"],
)

tools = {
    tool.name: tool
    for tool in file_toolkit.get_tools()
}

read_file = tools["read_file"]
write_file = tools["write_file"]

C:\Users\86183\AppData\Local\Temp\ipykernel_10812\69954654.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.agent_toolkits import FileManagementToolkit


In [10]:
write_file.invoke({
    "file_path": "test.py",
    "text": "print('hello')"
})

read_file.invoke({
    "file_path": "test.py"
})

"print('hello')"

In [12]:
# shell
from langchain_community.tools import ShellTool

shell = ShellTool(
    ask_human_input=True      # 请求
)

In [17]:
result = shell.invoke({
    "commands": ["python workspace/test.py"]
})

print(result)

Executing command:
 ['python workspace/test.py']
hello



# 3. RAG and rag tool

- build_index.py: build vector database.
- retriever: use Chroma to create a retriever
- rag tool

*Embedding*: 一个外面训练好的模型，将自然语言转化成向量

*vector database*: 包含 Embedding 以及一些向量数据，所以可以将输入的语句转化成向量，但不知道输出哪些数据

*retriever*: 给 vector db 加上检索机制

*rag_tool*: 包装 retriever 给 agent 使用

```text
RAG/
├── data/
│   └── Scan_User_Manual.pdf
│
├── vector_db/
│
├── rag/
│   ├── __init__.py
│   └── rag.py
│
└── build_index.py
```

## Embedding

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

def get_embeddings():
    """
    Create the local Qwen3 Embedding model.
    """

    embeddings = HuggingFaceEmbeddings(
        model_name="Qwen/Qwen3-Embedding-0.6B",

        model_kwargs={
            "device": "cpu",
        },

        encode_kwargs={
            "normalize_embeddings": True,
        },

        query_encode_kwargs={
            "prompt": (
                "Instruct: Given an EDA technical question, "
                "retrieve relevant passages from technical documentation "
                "that can help answer the question.\n"
                "Query: "
            ),
            "normalize_embeddings": True,
        },
    )

    return embeddings

embeddings = get_embeddings()

## vector db

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# 加载文件
PDF_PATH = "./data/Scan_User_Manual.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

# chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

# 建立 vector db
VECTOR_DB_DIR = "./vector_db"
COLLECTION_NAME = "eda_knowledge"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=VECTOR_DB_DIR,
)

## retriever

In [ ]:

VECTOR_DB_DIR = "./vector_db"
COLLECTION_NAME = "eda_knowledge"

# 导入 vector db
vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=VECTOR_DB_DIR,
    )

retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 4,
        },
    )

In [ ]:
query = "wrapper scan 是什么？"
docs = retriever.invoke(query)

for i, doc in enumerate(docs, start=1):

    print("=" * 60)

    print(f"Document {i}")

    print("Source:")
    print(doc.metadata.get("source"))

    print("Page:")
    print(doc.metadata.get("page"))

    print("Content:")
    print(doc.page_content)

## rag tool

In [ ]:
from langchain_core.tools import create_retriever_tool

rag_tool = create_retriever_tool(
        retriever=retriever,

        name="search_knowledge_base",

        description=(
            "Search the EDA technical knowledge base. "
            "Use this tool when the user asks about DFT, "
            "scan chain, wrapper scan, ScanInsertion, "
            "digital IC design, or other information that "
            "may be contained in the technical documents."
        ),
    )

In [ ]:
result = rag_tool.invoke({
    "query": "wrapper scan 和 boundary scan 有什么区别？"
})


print(result)

# 4. LangGraph: Agent loop

In [ ]:
# state
from langgraph.graph import MessagesState

State = MessagesState

In [ ]:
# node: llm_node, tool_node
from langchain.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import ToolNode

tools = [read_file, write_file, shell, rag_tool]

model_with_tool = model.bind_tools(tools)

# llm_node
SYSTEM_PROMPT = """你是一个 Agent。"""

def llm_node(state: State):
    response = model_with_tool.invoke(
        [
            SystemMessage(
                content=SYSTEM_PROMPT
            )
        ] + state["messages"]
    )

    return {
        "messages": [response]
    }

# tool node
tool_node = ToolNode(tools)

In [ ]:
# build graph
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition

builder = StateGraph(State)

builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)

builder.add_edge(START, "llm_node")

builder.add_conditional_edges(
    "llm_node",
    tools_condition,
    {
        "tool_node": "tool_node",
        END: END,
    },
)

builder.add_edge("tool_node", "llm_node")

graph = builder.compile()

In [ ]:
def run_agent_stream():

    print("=" * 60)
    print("Agent System")
    print("Type 'exit' to quit.")
    print("=" * 60)

    while True:

        user_input = input("\nUser: ").strip()

        if not user_input:
            continue

        if user_input.lower() == "exit":
            break

        try:

            for update in graph.stream(
                {
                    "messages": [
                        {
                            "role": "user",
                            "content": user_input,
                        }
                    ]
                },
                stream_mode="updates",
            ):

                for node_name, state_update in update.items():

                    print(f"\n========== {node_name} ==========")

                    messages = state_update.get(
                        "messages",
                        []
                    )

                    for message in messages:

                        # AIMessage
                        if hasattr(message, "tool_calls"):

                            if message.tool_calls:
                                print("Tool Calls:")

                                for tool_call in message.tool_calls:
                                    print(
                                        f"  - {tool_call['name']}"
                                    )
                                    print(
                                        f"    args: {tool_call['args']}"
                                    )

                            elif message.content:
                                print(
                                    "Assistant:",
                                    message.content
                                )

                        # ToolMessage
                        elif message.type == "tool":
                            print("Tool Result:")
                            print(message.content)

        except Exception as e:

            print("\nAgent Error:")
            print(e)

In [ ]:
def run_agent_invoke():

    print("=" * 60)
    print("Agent System")
    print("Type 'exit' to quit.")
    print("=" * 60)

    while True:

        user_input = input("\nUser: ").strip()

        if not user_input:
            continue

        if user_input.lower() == "exit":
            print("Bye!")
            break

        try:

            result = graph.invoke(
                {
                    "messages": [
                        {
                            "role": "user",
                            "content": user_input,
                        }
                    ]
                }
            )

            # 最后一条消息通常就是 Agent 的最终回答
            last_message = result["messages"][-1]

            print("\nAssistant:")
            print(last_message.content)

        except Exception as e:

            print("\nAgent Error:")
            print(e)